# Отчет по лабораторной работе №3: Классификация. Нейронные сети

**Дисциплина:** Системы искусственного интеллекта и машинное обучение  
**Тема:** Классификация и нейронные сети (Scikit-Learn, TensorFlow, TensorBoard)  
**Вариант:** Датасет характеристик мобильных устройств (`test.csv`)

## 1. Введение

**Цель работы:** изучить методы классификации данных, реализованные в библиотеке Scikit-Learn, и построить нейронную сеть на базе TensorFlow/Keras с использованием TensorBoard для мониторинга обучения.

В рамках данной лабораторной работы будут построены классификационные модели с помощью следующих методов:

1. Наивные байесовские классификаторы (`GaussianNB`, `MultinomialNB`, `ComplementNB`, `BernoulliNB`).
2. Дерево решений (`DecisionTreeClassifier`).
3. Линейный дискриминантный анализ (`LinearDiscriminantAnalysis`).
4. Метод опорных векторов (`SVC`).
5. Метод k-ближайших соседей (`KNeighborsClassifier`).

Для всех методов будут рассчитаны и сравнены метрики качества:

- Accuracy (точность);
- Precision (доля правильно классифицированных положительных примеров);
- Recall (чувствительность);
- F1-Score (гармоническое среднее precision и recall);
- Площадь под ROC-кривой (AUC-ROC).

Отдельный раздел посвящен построению нейронной сети на TensorFlow/Keras, исследованию влияния гиперпараметров и визуализации процесса обучения с помощью TensorBoard.


## 2. Выбор и подготовка датасета

В качестве исходного датасета используется файл **`test.csv`** из папки `data`, содержащий характеристики мобильных устройств (смартфонов): емкость батареи, наличие модулей связи (3G/4G, Wi-Fi, Bluetooth), объем оперативной памяти, разрешение экрана и другие технические параметры.

Так как в файле отсутствует явная целевая переменная (класс), в рамках данной лабораторной работы будет сформирован **искусственный целевой признак** на основе объема оперативной памяти `ram`:

- устройства с объемом RAM ниже медианы будут относиться к классу `0` ("бюджетный/средний сегмент"),
- устройства с объемом RAM не ниже медианы — к классу `1` ("производительный сегмент").

Таким образом, задача сводится к **бинарной классификации** смартфонов по их техническим характеристикам на два ценовых/производительных сегмента.

Далее будут выполнены:

- загрузка и первичный анализ данных;
- формирование целевого признака;
- разделение признаков и цели;
- масштабирование числовых признаков;
- разбиение на обучающую и тестовую выборки.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import shutil
import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.naive_bayes import GaussianNB, MultinomialNB, ComplementNB, BernoulliNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

sns.set(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", None)

np.random.seed(42)
tf.random.set_seed(42)


def prepare_log_dir(base_dir: str, model_name: str) -> str:
    """Подготовка директории логов TensorBoard (с очисткой старых логов)."""
    timestamp = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
    log_dir = os.path.join(base_dir, model_name, timestamp)
    if os.path.exists(log_dir):
        shutil.rmtree(log_dir)
    os.makedirs(log_dir, exist_ok=True)
    return log_dir


# Загрузка датасета test.csv
local_path = Path("../data/test.csv")
if local_path.exists():
    df = pd.read_csv(local_path)
    print(f"Данные загружены из локального файла: {local_path}")
else:
    # На случай запуска из другой директории
    df = pd.read_csv("data/test.csv")
    print("Данные загружены из файла data/test.csv")

print("Размер датасета:", df.shape)
print("Первые 5 строк датасета:")
display(df.head())

print("\nОбщая информация о датасете:")
print(df.info())

print("\nСтатистическое описание признаков:")
display(df.describe())

### 2.1 Формирование целевой переменной и предобработка данных

Сформируем бинарный целевой признак `target_ram_class` на основе объема оперативной памяти `ram`:

- `0` — устройство с объемом RAM ниже медианы по выборке;
- `1` — устройство с объемом RAM не ниже медианы.

Затем:

- удалим технический идентификатор `id` из признаков;
- проверим наличие пропусков и при необходимости обработаем их (в данном датасете пропусков нет);
- выполним стандартизацию признаков с помощью `StandardScaler` для дальнейшего использования в большинстве алгоритмов классификации.

In [ ]:
# Формирование целевой переменной по объему RAM
ram_median = df["ram"].median()
df["target_ram_class"] = (df["ram"] >= ram_median).astype(int)

print(f"Медиана RAM: {ram_median}")
print("Распределение целевого признака target_ram_class:")
print(df["target_ram_class"].value_counts(normalize=True))

# Проверка пропусков
print("\nКоличество пропущенных значений по столбцам:")
print(df.isnull().sum())

# Разделение на признаки и цель
feature_cols = [col for col in df.columns if col not in ["id", "target_ram_class"]]
X = df[feature_cols]
y = df["target_ram_class"]

print("\nСписок признаков:")
print(feature_cols)

# Масштабирование признаков
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns=feature_cols)

print("\nПервые 5 строк масштабированных признаков:")
display(X_scaled_df.head())

### 2.2 Разбиение выборки на обучающую и тестовую

Для оценки качества классификаторов разобьем данные на обучающую и тестовую части в пропорции 80/20 с сохранением баланса классов (`stratify`).

In [ ]:
# Разбиение выборки
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled_df, y, test_size=0.2, random_state=42, stratify=y
)

print("Размер обучающей выборки:", X_train.shape, y_train.shape)
print("Размер тестовой выборки:", X_test.shape, y_test.shape)

## 3. Метрики и вспомогательные функции оценки моделей

Для сравнения классификаторов будут использоваться следующие метрики:

- **Accuracy** — доля правильно классифицированных объектов;
- **Precision** — точность по положительному классу;
- **Recall** — полнота по положительному классу;
- **F1-score** — гармоническое среднее precision и recall;
- **AUC-ROC** — площадь под ROC-кривой.

Реализуем вспомогательную функцию, которая обучает модель, делает предсказания на тестовой выборке и возвращает словарь с рассчитанными метриками. Также напишем функцию для построения ROC-кривых для нескольких моделей.

In [ ]:
from typing import Dict, Any, List, Tuple


def evaluate_model(name: str, model, X_train, y_train, X_test, y_test) -> Dict[str, Any]:
    """Обучение модели и расчет основных метрик качества на тесте."""
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    metrics = {}
    metrics["model"] = name
    metrics["accuracy"] = accuracy_score(y_test, y_pred)
    metrics["precision"] = precision_score(y_test, y_pred, average="binary")
    metrics["recall"] = recall_score(y_test, y_pred, average="binary")
    metrics["f1"] = f1_score(y_test, y_pred, average="binary")

    # ROC-AUC (используем вероятности, если доступны)
    if hasattr(model, "predict_proba"):
        y_score = model.predict_proba(X_test)[:, 1]
        metrics["roc_auc"] = roc_auc_score(y_test, y_score)
    elif hasattr(model, "decision_function"):
        y_score = model.decision_function(X_test)
        metrics["roc_auc"] = roc_auc_score(y_test, y_score)
    else:
        metrics["roc_auc"] = np.nan

    return metrics


def plot_roc_curves(models: List[Tuple[str, Any]], X_test, y_test):
    """Построение ROC-кривых для нескольких обученных моделей."""
    plt.figure(figsize=(8, 6))

    for name, model in models:
        if hasattr(model, "predict_proba"):
            y_score = model.predict_proba(X_test)[:, 1]
        elif hasattr(model, "decision_function"):
            y_score = model.decision_function(X_test)
        else:
            continue

        fpr, tpr, _ = roc_curve(y_test, y_score)
        auc = roc_auc_score(y_test, y_score)
        plt.plot(fpr, tpr, label=f"{name} (AUC = {auc:.3f})")

    plt.plot([0, 1], [0, 1], "k--", label="Случайный классификатор")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate")
    plt.title("ROC-кривые для выбранных моделей")
    plt.legend(loc="lower right")
    plt.grid(True)
    plt.show()

## 4. Классические методы классификации (Scikit-Learn)

### 4.1 Наивные байесовские классификаторы

Рассмотрим четыре варианта наивного байесовского классификатора:

- `GaussianNB` — для непрерывных признаков, предполагаемых нормально распределенными;
- `MultinomialNB` — для неотрицательных счетчиков/частот;
- `ComplementNB` — модификация MultinomialNB, более устойчивая к несбалансированным классам;
- `BernoulliNB` — для бинарных признаков.

Для моделей Multinomial/Complement/Bernoulli используем масштабирование `MinMaxScaler` в конвейере `Pipeline`, чтобы признаки были неотрицательными и находились в сопоставимых диапазонах.

In [ ]:
# Наивные Байесовские классификаторы

models_nb = {
    "GaussianNB": GaussianNB(),
    "MultinomialNB": Pipeline([
        ("scaler", MinMaxScaler()),
        ("clf", MultinomialNB(alpha=1.0)),
    ]),
    "ComplementNB": Pipeline([
        ("scaler", MinMaxScaler()),
        ("clf", ComplementNB(alpha=1.0)),
    ]),
    "BernoulliNB": Pipeline([
        ("scaler", MinMaxScaler()),
        ("clf", BernoulliNB(alpha=1.0)),
    ]),
}

results_nb = []
trained_nb_models = []

for name, model in models_nb.items():
    print(f"\n=== {name} ===")
    metrics = evaluate_model(name, model, X_train, y_train, X_test, y_test)
    results_nb.append(metrics)
    trained_nb_models.append((name, model))
    print(metrics)

results_nb_df = pd.DataFrame(results_nb)
print("\nСводная таблица метрик (Наивные Байесовские классификаторы):")
display(results_nb_df)

# ROC-кривые для NB-моделей
plot_roc_curves(trained_nb_models, X_test, y_test)

### 4.2 Дерево решений, ЛДА, SVM и k-ближайших соседей

Далее рассмотрим следующие методы классификации:

- `DecisionTreeClassifier` — дерево решений;
- `LinearDiscriminantAnalysis` — линейный дискриминантный анализ;
- `SVC` (с ядром RBF и включенной вероятностной оценкой);
- `KNeighborsClassifier` — метод k-ближайших соседей.

Для большинства методов (кроме дерева решений) полезно использовать масштабирование признаков (`StandardScaler`), что уже было выполнено ранее.

In [ ]:
# Классические методы классификации

models_classic = {
    "DecisionTree": DecisionTreeClassifier(random_state=42),
    "LDA": LinearDiscriminantAnalysis(),
    "SVM_RBF": SVC(kernel="rbf", C=1.0, gamma="scale", probability=True, random_state=42),
    "KNN_5": KNeighborsClassifier(n_neighbors=5),
}

results_classic = []
trained_classic_models = []

for name, model in models_classic.items():
    print(f"\n=== {name} ===")
    metrics = evaluate_model(name, model, X_train, y_train, X_test, y_test)
    results_classic.append(metrics)
    trained_classic_models.append((name, model))
    print(metrics)

results_classic_df = pd.DataFrame(results_classic)
print("\nСводная таблица метрик (классические методы):")
display(results_classic_df)

# ROC-кривые для классических моделей
plot_roc_curves(trained_classic_models, X_test, y_test)

# Сводный DataFrame по всем моделям
all_results_df = pd.concat([results_nb_df, results_classic_df], ignore_index=True)
print("\nСводная таблица метрик по всем моделям:")
display(all_results_df)

plt.figure(figsize=(10, 5))
sns.barplot(data=all_results_df, x="model", y="accuracy")
plt.title("Сравнение accuracy моделей")
plt.xticks(rotation=45)
plt.show()

plt.figure(figsize=(10, 5))
sns.barplot(data=all_results_df, x="model", y="f1")
plt.title("Сравнение F1-score моделей")
plt.xticks(rotation=45)
plt.show()

## 5. Настройка гиперпараметров

В этом разделе будут проведены небольшие эксперименты по настройке гиперпараметров для различных методов классификации и оценено влияние параметров на качество (accuracy / F1 / AUC-ROC).

Для примера:

- для наивных байесовских моделей будет изменяться параметр `alpha`;
- для дерева решений — глубина `max_depth`;
- для SVM — параметры `C` и `gamma`;
- для k-ближайших соседей — число соседей `n_neighbors`.

In [ ]:
# Пример настройки alpha для MultinomialNB

alphas = [0.1, 0.5, 1.0, 2.0]
nb_alpha_results = []

for a in alphas:
    model = Pipeline([
        ("scaler", MinMaxScaler()),
        ("clf", MultinomialNB(alpha=a)),
    ])
    metrics = evaluate_model(f"MultinomialNB_alpha_{a}", model, X_train, y_train, X_test, y_test)
    nb_alpha_results.append({"alpha": a, "accuracy": metrics["accuracy"], "f1": metrics["f1"], "roc_auc": metrics["roc_auc"]})

nb_alpha_df = pd.DataFrame(nb_alpha_results)
print("Влияние параметра alpha в MultinomialNB:")
display(nb_alpha_df)

plt.figure(figsize=(8, 4))
plt.plot(nb_alpha_df["alpha"], nb_alpha_df["accuracy"], marker="o", label="accuracy")
plt.plot(nb_alpha_df["alpha"], nb_alpha_df["f1"], marker="o", label="f1")
plt.xlabel("alpha")
plt.ylabel("Метрика")
plt.title("Влияние alpha на качество MultinomialNB")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Настройка глубины дерева решений

max_depth_values = [3, 5, 7, None]
tree_results = []

for d in max_depth_values:
    model = DecisionTreeClassifier(max_depth=d, random_state=42)
    metrics = evaluate_model(f"DecisionTree_depth_{d}", model, X_train, y_train, X_test, y_test)
    tree_results.append({"max_depth": str(d), "accuracy": metrics["accuracy"], "f1": metrics["f1"], "roc_auc": metrics["roc_auc"]})

tree_df = pd.DataFrame(tree_results)
print("Влияние max_depth в DecisionTree:")
display(tree_df)

plt.figure(figsize=(8, 4))
plt.plot(tree_df["max_depth"], tree_df["accuracy"], marker="o", label="accuracy")
plt.plot(tree_df["max_depth"], tree_df["f1"], marker="o", label="f1")
plt.xlabel("max_depth")
plt.ylabel("Метрика")
plt.title("Влияние max_depth на качество дерева решений")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
# Небольшой Grid Search для SVM по параметрам C и gamma

param_grid_svm = {
    "C": [0.1, 1, 10],
    "gamma": [0.01, 0.1, 1.0],
}

svm = SVC(kernel="rbf", probability=True, random_state=42)

svm_grid = GridSearchCV(
    estimator=svm,
    param_grid=param_grid_svm,
    scoring="f1",
    cv=3,
    n_jobs=-1,
)

svm_grid.fit(X_train, y_train)

print("Лучшие параметры SVM:", svm_grid.best_params_)
print("Лучшее значение F1 (CV):", svm_grid.best_score_)

svm_best = svm_grid.best_estimator_
svm_best_metrics = evaluate_model("SVM_RBF_best", svm_best, X_train, y_train, X_test, y_test)
print("Метрики лучшей модели SVM на тесте:")
print(svm_best_metrics)

In [ ]:
# Настройка числа соседей для KNN

neighbors = [3, 5, 7, 9]
knn_results = []

for k in neighbors:
    model = KNeighborsClassifier(n_neighbors=k)
    metrics = evaluate_model(f"KNN_{k}", model, X_train, y_train, X_test, y_test)
    knn_results.append({"n_neighbors": k, "accuracy": metrics["accuracy"], "f1": metrics["f1"], "roc_auc": metrics["roc_auc"]})

knn_df = pd.DataFrame(knn_results)
print("Влияние числа соседей в KNN:")
display(knn_df)

plt.figure(figsize=(8, 4))
plt.plot(knn_df["n_neighbors"], knn_df["accuracy"], marker="o", label="accuracy")
plt.plot(knn_df["n_neighbors"], knn_df["f1"], marker="o", label="f1")
plt.xlabel("n_neighbors")
plt.ylabel("Метрика")
plt.title("Влияние числа соседей на качество KNN")
plt.legend()
plt.grid(True)
plt.show()

## 6. Нейронная сеть на TensorFlow/Keras

В этом разделе будет реализована простая полносвязная нейронная сеть для бинарной классификации `target_ram_class`.

Будет рассмотрена базовая архитектура:

- входной слой размерности `n_features`;
- один–два скрытых слоя с активацией ReLU;
- выходной слой с активацией Sigmoid.

В качестве функции потерь используется `binary_crossentropy`, оптимизатор — `Adam`. Для мониторинга обучения (loss/accuracy на train/validation) будет использоваться **TensorBoard**.

Также проведем небольшой эксперимент по изменению гиперпараметров (число нейронов и скорость обучения) и сравним полученные метрики.

In [ ]:
input_dim = X_train.shape[1]


def build_dense_model(units: int = 32, learning_rate: float = 1e-3) -> keras.Model:
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(units, activation="relu"),
        layers.Dense(units // 2, activation="relu"),
        layers.Dense(1, activation="sigmoid"),
    ])

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss="binary_crossentropy",
        metrics=["accuracy", tf.keras.metrics.AUC(name="auc")],
    )
    return model


# Базовая модель
base_model = build_dense_model(units=32, learning_rate=1e-3)

log_dir_base = prepare_log_dir("logs_tf", "dense_base")
tensorboard_cb = keras.callbacks.TensorBoard(log_dir=log_dir_base, histogram_freq=1)
early_stopping_cb = keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True, monitor="val_loss")

history_base = base_model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=50,
    batch_size=32,
    callbacks=[tensorboard_cb, early_stopping_cb],
    verbose=0,
)

# Оценка на тесте
nn_test_loss, nn_test_acc, nn_test_auc = base_model.evaluate(X_test, y_test, verbose=0)
print(f"Базовая нейронная сеть — accuracy: {nn_test_acc:.4f}, AUC: {nn_test_auc:.4f}")

# Графики обучения
plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
plt.plot(history_base.history["loss"], label="train_loss")
plt.plot(history_base.history["val_loss"], label="val_loss")
plt.xlabel("Эпоха")
plt.ylabel("Loss")
plt.title("График функции потерь (NN)")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history_base.history["accuracy"], label="train_acc")
plt.plot(history_base.history["val_accuracy"], label="val_acc")
plt.xlabel("Эпоха")
plt.ylabel("Accuracy")
plt.title("График точности (NN)")
plt.legend()

plt.tight_layout()
plt.show()

print(f"Логи TensorBoard сохранены в: {log_dir_base}")
print("Запуск TensorBoard (из командной строки): tensorboard --logdir=logs_tf")

In [ ]:
# Небольшой эксперимент по гиперпараметрам нейронной сети

configs = [
    {"units": 16, "lr": 1e-3},
    {"units": 32, "lr": 1e-3},
    {"units": 64, "lr": 5e-4},
]

nn_results = []

for cfg in configs:
    print(f"\nОбучение модели: units={cfg['units']}, lr={cfg['lr']}")
    model = build_dense_model(units=cfg["units"], learning_rate=cfg["lr"])
    log_dir = prepare_log_dir("logs_tf", f"dense_u{cfg['units']}_lr{cfg['lr']}")
    tb_cb = keras.callbacks.TensorBoard(log_dir=log_dir, histogram_freq=0)

    history = model.fit(
        X_train,
        y_train,
        validation_split=0.2,
        epochs=40,
        batch_size=32,
        callbacks=[tb_cb, early_stopping_cb],
        verbose=0,
    )

    loss, acc, auc = model.evaluate(X_test, y_test, verbose=0)
    nn_results.append(
        {
            "units": cfg["units"],
            "lr": cfg["lr"],
            "accuracy": acc,
            "auc": auc,
        }
    )

nn_results_df = pd.DataFrame(nn_results)
print("\nРезультаты эксперимента с гиперпараметрами нейронной сети:")
display(nn_results_df)

## 7. Сравнительный анализ моделей

В этом разделе можно свести результаты:

- классических методов (Наивные Байесовские, дерево решений, ЛДА, SVM, KNN);
- нейронной сети TensorFlow.

В отдельной таблице будут представлены accuracy, F1-score и AUC-ROC для всех рассмотренных моделей, после чего сформулируем выводы о лучшем подходе для данного датасета.

In [ ]:
# Добавим в сводную таблицу результат базовой нейронной сети

nn_summary = pd.DataFrame([
    {
        "model": "TensorFlow_DNN_base",
        "accuracy": nn_test_acc,
        "precision": np.nan,  # при желании можно досчитать по порогу 0.5
        "recall": np.nan,
        "f1": np.nan,
        "roc_auc": nn_test_auc,
    }
])

final_results_df = pd.concat([all_results_df, nn_summary], ignore_index=True)
print("Сравнительная таблица метрик для всех моделей:")
display(final_results_df)

plt.figure(figsize=(10, 5))
sns.barplot(data=final_results_df, x="model", y="accuracy")
plt.title("Сравнение accuracy всех моделей (включая нейронную сеть)")
plt.xticks(rotation=45)
plt.show()

plt.figure(figsize=(10, 5))
sns.barplot(data=final_results_df, x="model", y="roc_auc")
plt.title("Сравнение AUC-ROC всех моделей")
plt.xticks(rotation=45)
plt.show()

## 8. Заключение

В ходе лабораторной работы были изучены и реализованы различные методы классификации (Наивные Байесовские классификаторы, дерево решений, ЛДА, SVM, k-ближайших соседей) на датасете характеристик мобильных устройств `test.csv`. Для всех моделей были рассчитаны метрики качества (accuracy, precision, recall, F1-score, AUC-ROC) и проведены эксперименты по настройке гиперпараметров.

Была построена и обучена нейронная сеть на TensorFlow/Keras для бинарной классификации устройств по классу `target_ram_class`, продемонстрировано использование TensorBoard для мониторинга процесса обучения и влияние изменения гиперпараметров (число нейронов, скорость обучения) на финальное качество.

Сравнительный анализ показал, что (на основе полученных метрик в таблице) наилучшие результаты на данном датасете достигаются методами, которые вы можете указать в отчете после запуска ноутбука (например, SVM или нейронная сеть), что подчеркивает важность настройки параметров моделей и выбора подходящего алгоритма для конкретной задачи.

## 9. Список источников

1. Документация Scikit-Learn: разделы по `Naive Bayes`, `DecisionTreeClassifier`, `LinearDiscriminantAnalysis`, `SVC`, `KNeighborsClassifier`.
2. Документация TensorFlow/Keras: руководство по построению моделей Sequential и использованию TensorBoard.
3. Учебные материалы по курсу "Системы искусственного интеллекта и машинное обучение".

## 10. Приложение

Полный листинг программного кода приведен в данном Jupyter-ноутбуке по разделам лабораторной работы.